# Full synthetic date-normalization experiment

Train an explicit encoder-decoder Transformer on 4,000 unique dates, select on 400 validation dates, then score the reloaded checkpoint on 400 separate test dates. All splits share the same English format and 1950–2050 range. A deterministic rule parser is also evaluated; this is not general machine translation.

In [ ]:
import os
from pathlib import Path
import uuid
project = Path.cwd()
if not (project/'configs').is_dir(): project = project.parent
assert (project/'configs').is_dir(), 'Launch from the project or notebooks folder'
os.chdir(project)
run = project/'artifacts'/('notebook-'+uuid.uuid4().hex[:12])

In [ ]:
from date_transformer.workflow import train,predict,evaluate_checkpoint
result = train(project/'configs/dates-cpu.json',run)
print('Selected epoch:',result['selected_epoch'])
print('Validation:',result['validation'])
print('Held-out test:',result['test'])

In [ ]:
from IPython.display import display
from PIL import Image
display(Image.open(run/'training.png'))
display(Image.open(run/'attention.png'))

## Greedy decoding and explicit EOS attention rows

The domain parser only validates input; it never substitutes a correct ISO value for a neural prediction. All emitted tokens and the last-layer mean attention weights are retained.

In [ ]:
prediction = predict(run/'best.pt','August 3, 2026',run/'example',attention=True)
print('Predicted:',prediction['prediction'])
print('EOS:',prediction['terminated'],'Valid ISO:',prediction['valid_iso'])
assert len(prediction['attention']) == len(prediction['attention_rows'])
assert all(len(row)==len(prediction['attention_columns']) for row in prediction['attention'])
display(Image.open(run/'example/attention.png'))

In [ ]:
reloaded = evaluate_checkpoint(run/'best.pt',run/'reloaded-test')
assert reloaded == result['test']
print('Reloaded held-out evaluation matches the training report.')

High accuracy on one fixed synthetic format does not establish out-of-domain correctness. The rule parser solves this task without training. Attention is descriptive, not a causal explanation. Published result JSON and per-test predictions document the measured run; values may vary across platforms/versions.